# GWAS vs Linear Mixed Models: A Stress-Test Comparison Using 1000 Genomes chr22

This report summarizes the pipeline and findings from **main.ipynb**, which evaluates how genome-wide association (GWA) testing behaves under several deliberately challenging conditions (“stress scenarios”).

---

## Hypothesis

We hypothesize that **association test calibration (type I error and p-value validity) is sensitive to**:

1. **Population structure** — when the phenotype is correlated with ancestry (e.g., PC1), linear models without covariates may inflate test statistics.  
2. **Case–control imbalance** — binary traits with extreme case fractions may affect logistic regression stability and calibration.  
3. **Sample relatedness** — in highly related cohorts, linear mixed models (LMMs) may still show inflation if the phenotype aligns with genetic structure; in low-related cohorts, over-correction may cause deflation.  
4. **Genotype missingness** — randomly missing genotypes can bias effect estimates and p-values when not properly accounted for.

We test these by **simulating phenotypes** under controlled models and then running **PLINK (linear/logistic) and GEMMA (LMM)** association, and assessing **λGC and QQ plots** across scenarios.

---

## Introduction


Genome-wide association studies (GWAS) are widely used to identify genetic variants associated with complex traits. However, standard association models such as linear or logistic regression can become miscalibrated when important assumptions are violated. Factors such as population structure, sample relatedness, case–control imbalance, and genotype missingness may lead to inflated or deflated test statistics.

In this project we build an analysis pipeline to systematically evaluate how these factors affect GWAS calibration. Using genotype data from chromosome 22 of the 1000 Genomes Project, we simulate phenotypes under controlled models and evaluate association results using both standard GWAS (PLINK) and linear mixed models (GEMMA). We design several “stress scenarios” that intentionally violate GWAS assumptions and quantify their impact using metrics such as genomic inflation factor (λGC) and QQ plots.

This analysis provides insight into when simple GWAS models fail and how mixed models or covariate correction can 
improve calibration.

All phenotypes are simulated from **real chr22 genotypes** (QC’d, 97,216 SNPs; 2,504 individuals). The base model is:

$$
y = G_{\text{causal}} \beta + C + \epsilon
$$

where $G_{\text{causal}}$ is a set of randomly chosen causal SNPs, $\beta$ are random effects, $C$ is a “stress” component, and $\epsilon$ is Gaussian noise. We define four stress scenarios:

| SS | Name | Description |
|----|------|-------------|
| **SS1** | Population structure confounding | $C = a \cdot \text{PC1}$; we vary $a \in \{0, 0.5, 1, 2, 3\}$ to inject ancestry-related signal. |
| **SS2** | Case–control imbalance | Binary trait: $\text{logit}(P(\text{case})) = G\beta + \alpha$; we vary $\alpha$ to get case proportions from ~34% to ~99%. |
| **SS3** | Sample relatedness stress | Two cohorts: **high-related** (200 individuals with strongest kinship) and **low-related** (200 with minimal kinship). Phenotype $y = \text{PC1} + \epsilon$; chr22 LMM run per cohort. |
| **SS4** | Genotype missingness | Randomly mask a fraction of genotype entries (e.g., 1%, 5%, 10%, 20%), then re-run simple linear association (y ~ SNP) and compare p-value distributions. |

Baseline (no stress): $C = 0$, continuous $y = G_{\text{causal}}\beta + \epsilon$; used for PLINK linear (with/without PC covariates) and for comparison with stressed runs.

---


## Methods

### Data and preprocessing (main.ipynb §§1–3)

Genotype data were obtained from the **1000 Genomes Project** and processed using standard GWAS preprocessing steps. VCF files for **chromosomes 20–22** were converted to **PLINK binary format (bed/bim/fam)** for efficient analysis.

Quality control filters were applied to remove unreliable variants:

- Minor allele frequency (MAF) ≥ 0.05  
- Genotype missingness ≤ 0.02  
- Hardy–Weinberg equilibrium \(p \le 10^{-6}\)

To reduce redundancy from linkage disequilibrium (LD), we performed **LD pruning** using PLINK with parameters:

--indep-pairwise 50 5 0.2


Two datasets were created for different parts of the analysis:

**Genome-wide dataset (chr20–22).**  
This dataset was used to estimate population structure and relatedness. Principal component analysis (PCA) was performed to compute the top **10 principal components**, and the first **five PCs** were used as covariates in association testing (`covariates.txt`). The same dataset was also used to compute a **genetic relationship matrix (GRM)** using GEMMA (`-gk 1`). After LD pruning, this dataset contained about **21k SNPs across 2,504 individuals**.

**chr22-only dataset.**  
Association tests were performed only on chromosome 22 variants to reduce computational cost while keeping realistic genomic structure. After QC, this dataset contained **97,216 SNPs**.

---

### Phenotype simulation (§4)


We simulated several phenotype scenarios to test how association methods behave under different conditions.

- **SS1:** Continuous $y = G\beta + a\cdot\text{PC1} + \epsilon$; outputs in `output/pheno_pc_confound/`.
- **SS2:** Binary from logistic model; outputs in `output/pheno_casecontrol/`.
- **SS3:** Phenotype = PC1 + small noise for highrel/lowrel subsets; GRM and LMM run per cohort.
- **SS4:** Same baseline phenotype; genotypes masked at given missing rates; per-SNP linear regression, results in `output/assoc_missingness/`.

---

### Association testing (main.ipynb §5)

Two association frameworks were compared.

**PLINK regression**

- Linear regression for continuous phenotypes
- Logistic regression for binary phenotypes
- Models run **with and without PC covariates**

Output files:

- `plink_nocov`
- `plink_pc`
- `plink_logistic`

**GEMMA mixed model**

Linear mixed models (LMM) were run using GEMMA with -lmm 4


The GRM was included to account for genetic relatedness.  
Separate models were run for the **high-relatedness** and **low-relatedness** cohorts.

---

### Calibration and visualization (main.ipynb §6)

Association results were evaluated using standard GWAS diagnostics.

**Genomic inflation factor**

\[
\lambda_{GC} = \frac{\text{median}(\chi^2_1)}{0.456}
\]

Values close to 1 indicate well-calibrated statistics.

**QQ plots**

QQ plots compare observed and expected \(-\log_{10}(p)\) values to visualize inflation or deflation of test statistics.

**Summary metrics**

For each scenario we recorded:

- genomic inflation factor (λGC)
- number of significant hits
- overlap of top variants between PLINK and GEMMA.

## Results

### Baseline GWAS calibration

We first evaluated the baseline association results using standard linear regression in PLINK.  
The genomic inflation factor (λGC) is close to 1, indicating that the test statistics follow the expected null distribution when no strong confounding factors are present.

Adding principal component (PC) covariates produces nearly identical results, suggesting that the simulated baseline phenotype does not contain strong population structure effects.

---

### S1 – Baseline GWAS calibration

We first evaluated the calibration of association statistics using the genomic inflation factor (λGC).  
Large λGC values indicate inflation of test statistics, while values close to 1 indicate well-calibrated results.

The PLINK model without covariates shows extreme inflation (λGC ≈ 22.7), suggesting strong confounding effects.  
After including principal component covariates, the inflation is dramatically reduced (λGC ≈ 1.15), indicating that population structure explains much of the bias.

For the mixed model analysis, the high-relatedness cohort still shows moderate inflation (λGC ≈ 1.27), while the low-relatedness cohort shows slight deflation (λGC ≈ 0.78).  
This suggests that sample relatedness can affect calibration even when using mixed models.

| method | dataset | n_valid_p | λGC |
|---|---|---:|---:|
| GEMMA | high-related cohort | 97216 | 1.27 |
| GEMMA | low-related cohort | 97216 | 0.78 |
| PLINK linear | no covariates | 97213 | 22.72 |
| PLINK linear | PC covariates | 97214 | 1.15 |

**Table 1.** Genomic inflation factor (λGC) for different association models.

The λGC values were computed from association p-values using the standard definition:

\[
\lambda_{GC} = \frac{\text{median}(\chi^2_1)}{0.456}
\]

The full summary file is available at:


---

### SS2 – Case–control phenotype

For the binary phenotype scenario, logistic regression was used to perform association testing.  
The resulting QQ plot remains mostly well calibrated but shows slightly greater variability in the tail of the distribution compared to the continuous phenotype model.

This behavior is expected since logistic regression with limited signal and class imbalance can produce more variable p-value distributions.

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; max-width: 700px;">

<img src="output/section6/figs/qq_plink_nocov_assoc_linear.png" width="100%">
<img src="output/section6/figs/qq_plink_pc_assoc_linear.png" width="100%">

<img src="output/section6/figs/qq_assoc_highrel_assoc_txt.png" width="100%">
<img src="output/section6/figs/qq_assoc_lowrel_assoc_txt.png" width="100%">

</div>

**Figure 1.** QQ plots comparing association methods.  
Top row: PLINK without covariates vs PLINK with PC covariates.  
Bottom row: GEMMA mixed model results for high-relatedness and low-relatedness cohorts.

---

### SS3 – Relatedness effects

To study the effect of relatedness, we compared standard PLINK regression with GEMMA linear mixed models (LMM). The QQ plots show that inflation is stronger in settings with higher relatedness, while the mixed model remains better calibrated.

To further compare the two methods, we summarized overlap in significant SNPs and top-ranked SNPs between PLINK and GEMMA. Without covariates, the overlap is low: for example, at the genome-wide significance threshold (\(5\times10^{-8}\)), the Jaccard index is only 0.025, and the top-50 overlap Jaccard is 0.266. After adding PC covariates, the agreement improves substantially: the significant-SNP Jaccard increases to 0.573 and the top-50 Jaccard increases to 0.887.

This suggests that uncorrected PLINK results are strongly affected by confounding, while PC adjustment makes the fixed-effect model much more consistent with the mixed model. In other words, relatedness-aware or structure-aware correction is important not only for calibration, but also for reproducibility of the strongest association signals.

| comparison | threshold_p | n_sig_plink | n_sig_gemma | n_overlap | jaccard |
|---|---:|---:|---:|---:|---:|
| baseline_nocov | 5e-8 | 23613 | 959 | 603 | 0.025 |
| baseline_nocov | 1e-5 | 33643 | 1257 | 741 | 0.022 |
| baseline_nocov | 1e-3 | 47610 | 1789 | 1130 | 0.023 |
| baseline_pc | 5e-8 | 921 | 905 | 665 | 0.573 |
| baseline_pc | 1e-5 | 1382 | 1192 | 834 | 0.479 |
| baseline_pc | 1e-3 | 2177 | 1607 | 1040 | 0.379 |

**Table 2.** Overlap in significant SNPs between PLINK and GEMMA under different significance thresholds.

| comparison | topK | overlap_topK | jaccard_topK |
|---|---:|---:|---:|
| baseline_nocov | 50 | 21 | 0.266 |
| baseline_nocov | 100 | 53 | 0.361 |
| baseline_nocov | 500 | 192 | 0.238 |
| baseline_pc | 50 | 47 | 0.887 |
| baseline_pc | 100 | 84 | 0.724 |
| baseline_pc | 500 | 363 | 0.570 |

**Table 3.** Overlap in top-ranked SNPs between PLINK and GEMMA.



---
### S4 – Agreement between PLINK and GEMMA

To quantify the overall agreement between association methods, we computed the Spearman correlation between p-values produced by PLINK and GEMMA across all SNPs.

Without covariate adjustment (`baseline_nocov`), the correlation between the two methods is very low (ρ = 0.053). This indicates that the ranking of SNP significance differs substantially between the two models when population structure is not corrected.

After including principal component covariates (`baseline_pc`), the agreement improves noticeably (ρ = 0.349). Although the correlation is still moderate, the improvement suggests that correcting for population structure makes the fixed-effect PLINK model more consistent with the mixed-model results.
<table style="border: none;">
<tr>
<td style="border: none;">
<img src="output/section6/figs/corr_baseline_nocov.png" width="320">
</td>

<td style="border: none;">
<img src="output/section6/figs/corr_baseline_pc.png" width="320">
</td>
</tr>
</table>

**Figure 4.** Correlation of SNP association statistics between PLINK and GEMMA.  
Left: baseline without covariates (ρ = 0.053).  
Right: baseline with PC covariates (ρ = 0.349).

These results support the previous overlap analysis: uncorrected regression produces association rankings that differ substantially from mixed-model results, while PC adjustment improves consistency between the two approaches.

---

### Summary of calibration metrics

Across all scenarios, the genomic inflation factor varied depending on how strongly GWAS assumptions were violated.

Population structure and sample relatedness produced the largest inflation, while genotype missingness mainly reduced stability rather than causing systematic bias.

A summary table of λGC values for all experiments was generated in:


## Discussion

Our experiments highlight how violations of standard GWAS assumptions can strongly affect association results. In particular, population structure and sample relatedness introduce substantial bias when not properly modeled.

The baseline results demonstrate that uncorrected linear regression can produce severely inflated statistics. The extremely large genomic inflation factor (λGC ≈ 22.7) in the PLINK model without covariates indicates that population structure can dramatically distort association signals. Adding principal component covariates substantially reduces inflation, suggesting that ancestry-related variation is a major confounding factor.

Relatedness also affects association calibration. While linear mixed models partially correct for hidden relatedness by incorporating the genetic relationship matrix, differences between PLINK and GEMMA remain noticeable. This is reflected not only in λGC values but also in the overlap and correlation analyses. Without covariates, the strongest signals discovered by PLINK and GEMMA differ substantially. After PC adjustment, the agreement between the two methods improves considerably.

The overlap analysis further shows that confounding does not only affect calibration metrics but also alters which variants appear significant. When covariates are included, the Jaccard similarity between the top-ranked SNPs increases dramatically, suggesting that proper correction improves reproducibility of GWAS discoveries.

Overall, these results demonstrate that controlling for population structure and relatedness is critical for reliable GWAS analysis. Simple regression models can perform reasonably well when appropriate covariates are included, but mixed models provide a more principled framework for handling complex sample relationships.

### Limitations and challenges

Several limitations should be noted. First, our experiments were conducted on chromosome 22 only to reduce computational cost. While this allows faster experimentation, it does not fully reflect genome-wide analyses with millions of variants.

Second, the phenotypes used in this study were simulated. Although this allows controlled evaluation of confounding effects, real biological phenotypes may involve more complex genetic architectures.

Finally, our analysis focused primarily on calibration and agreement between methods rather than biological interpretation of specific variants. Future work could extend this framework to real phenotypes and evaluate how these modeling choices affect downstream biological conclusions.

### Future directions

Several extensions could further improve this analysis. One direction is to apply the same evaluation framework to additional GWAS methods such as BOLT-LMM or REGENIE, which are widely used for large-scale association studies.

Another extension would be to perform genome-wide analyses using the full 1000 Genomes dataset to better evaluate how these methods scale in larger studies.

Finally, incorporating real phenotypes rather than simulated traits would allow investigation of how population structure and relatedness affect real biological signals.

## Code Availability

All scripts used for data preprocessing, phenotype simulation, association testing, and analysis are available in the project GitHub repository:

https://github.com/jhhuangchloe/CSE284

The repository includes the main analysis notebook (`main.ipynb`), preprocessing scripts, and generated summary statistics used in this report.